# BirdCLEF 2026 — Perch v4 Test Soundscape Cache

Pre-compute Perch embeddings + competition-space probs for all test soundscapes.
Output is attached as a Kaggle dataset and loaded by the fast inference kernel,
which skips Perch entirely and runs only probes + post-processing.

Output: `/kaggle/working/test_perch_cache.npz`
  - `row_ids`:   (N_slots,) str   — '{stem}_{end_sec}'
  - `stems`:     (N_slots,) str   — soundscape stem for grouping
  - `embeddings`: (N_slots, 1280) float32
  - `comp_probs`: (N_slots, 234)  float32  sigmoid probs in competition space
  - `species`:   (234,) str

In [ ]:
import os
import time

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# ── paths ─────────────────────────────────────────────────────────────────────
DATASETS = "/kaggle/input/datasets/aldisued"
COMP_DIR = "/kaggle/input/competitions/birdclef-2026"
PERCH_MODEL_DIR = (
    f"{DATASETS}/perch-v4-model/tfhub_modules/5dcbb82658655292c50ca88ce1e6f1073b17d0d9"
)
PERCH_LABEL_CSV = f"{PERCH_MODEL_DIR}/assets/label.csv"
OUT_NPZ = "/kaggle/working/test_perch_cache.npz"

for p in [COMP_DIR, PERCH_MODEL_DIR]:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"{p.split('/')[-1]}: {status}")

In [ ]:
# ── load Perch v4 ─────────────────────────────────────────────────────────────
print("Loading Perch v4 SavedModel...")
t0 = time.time()
perch_model = tf.saved_model.load(PERCH_MODEL_DIR)
print(f"Loaded in {time.time() - t0:.1f}s")

test_audio = tf.zeros([1, 32000 * 5], dtype=tf.float32)
logits_test, emb_test = perch_model.infer_tf(test_audio)
print(f"Smoke test — logits: {logits_test.shape}, embeddings: {emb_test.shape}")

In [ ]:
# ── species + Perch label mapping ────────────────────────────────────────────
taxonomy = pd.read_csv(os.path.join(COMP_DIR, "taxonomy.csv"))
competition_species = taxonomy["primary_label"].astype(str).tolist()
n_species = len(competition_species)
print(f"Competition species: {n_species}")

with open(PERCH_LABEL_CSV) as f:
    perch_labels = [line.strip() for line in f if line.strip()]
n_perch = len(perch_labels)
perch_to_idx = {lbl: i for i, lbl in enumerate(perch_labels)}
print(f"Perch v4 labels: {n_perch}")

comp_to_perch = np.array(
    [perch_to_idx.get(sp, -1) for sp in competition_species], dtype=np.int32
)
perch_coverage = comp_to_perch >= 0
print(f"Direct Perch mapping: {perch_coverage.sum()} / {n_species}")

# Genus proxy for unmapped species
genus_to_perch_indices = {}
for i, lbl in enumerate(perch_labels):
    prefix = lbl[:3].lower()
    genus_to_perch_indices.setdefault(prefix, []).append(i)

taxonomy["genus"] = taxonomy["scientific_name"].str.split().str[0].str.lower()
sp_genus_perch = {}
for _, row in taxonomy.iterrows():
    sp = str(row["primary_label"])
    if comp_to_perch[competition_species.index(sp)] >= 0:
        continue
    genus = row["genus"]
    prefix = genus[:3].lower() if len(genus) >= 3 else genus.lower()
    indices = genus_to_perch_indices.get(prefix, [])
    if indices:
        sp_genus_perch[sp] = indices

print(f"Genus proxy: {len(sp_genus_perch)} unmapped species")

EMB_DIM = emb_test.shape[-1]  # 1280

In [ ]:
# ── inference + cache loop ────────────────────────────────────────────────────
SR = 32_000
CLIP_SAMPLES = SR * 5
SAVE_EVERY = 50  # checkpoint every N files


def logits_to_comp_probs(raw_logits: np.ndarray) -> np.ndarray:
    """(B, n_perch) logits → (B, n_species) sigmoid probs."""
    cp = np.zeros((len(raw_logits), n_species), dtype=np.float32)
    cp[:, perch_coverage] = 1.0 / (
        1.0 + np.exp(-raw_logits[:, comp_to_perch[perch_coverage]])
    )
    for sp, indices in sp_genus_perch.items():
        sp_idx = competition_species.index(sp)
        max_logit = raw_logits[:, indices].max(axis=1)
        cp[:, sp_idx] = 1.0 / (1.0 + np.exp(-max_logit))
    return cp


# Find test soundscapes
import pathlib

sc_dir = pathlib.Path(COMP_DIR) / "test_soundscapes"
test_files = sorted(sc_dir.iterdir()) if sc_dir.exists() else []
print(f"Test soundscapes: {len(test_files)}")

all_row_ids: list[str] = []
all_stems: list[str] = []
all_embeddings: list[np.ndarray] = []
all_comp_probs: list[np.ndarray] = []

t_start = time.time()
n_errors = 0

for i, sc_path in enumerate(test_files):
    stem = sc_path.stem
    t_file = time.time()

    try:
        y, _ = librosa.load(str(sc_path), sr=SR, mono=True)
    except Exception as e:
        print(f"WARN: {sc_path.name}: {e}")
        n_errors += 1
        continue

    n_slots = len(y) // CLIP_SAMPLES
    if n_slots == 0:
        continue

    chunks = np.stack(
        [y[s * CLIP_SAMPLES : (s + 1) * CLIP_SAMPLES] for s in range(n_slots)]
    )
    x = tf.constant(chunks, dtype=tf.float32)
    raw_logits, embs = perch_model.infer_tf(x)
    raw_logits = raw_logits.numpy()
    embs = embs.numpy()  # (n_slots, 1280)

    cp = logits_to_comp_probs(raw_logits)  # (n_slots, 234)

    for slot in range(n_slots):
        end_sec = (slot + 1) * 5
        all_row_ids.append(f"{stem}_{end_sec}")
        all_stems.append(stem)
        all_embeddings.append(embs[slot])
        all_comp_probs.append(cp[slot])

    t_file = time.time() - t_file
    if (i + 1) % 20 == 0 or i == 0:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        eta = (len(test_files) - i - 1) / max(rate, 1e-9)
        print(
            f"[{i + 1}/{len(test_files)}] {rate:.2f} files/s | "
            f"file={t_file:.1f}s | ETA {eta / 60:.1f}min | errors={n_errors}",
            flush=True,
        )

    if (i + 1) % SAVE_EVERY == 0:
        np.savez_compressed(
            OUT_NPZ,
            row_ids=np.array(all_row_ids),
            stems=np.array(all_stems),
            embeddings=np.stack(all_embeddings),
            comp_probs=np.stack(all_comp_probs),
            species=np.array(competition_species),
        )
        print(f"Checkpoint: {len(all_row_ids)} slots saved", flush=True)

elapsed = time.time() - t_start
print(f"\nDone in {elapsed / 60:.1f} min | errors: {n_errors}")
print(f"Total slots: {len(all_row_ids)}")

In [ ]:
# ── final save ────────────────────────────────────────────────────────────────
np.savez_compressed(
    OUT_NPZ,
    row_ids=np.array(all_row_ids),
    stems=np.array(all_stems),
    embeddings=np.stack(all_embeddings)
    if all_embeddings
    else np.zeros((0, EMB_DIM), dtype=np.float32),
    comp_probs=np.stack(all_comp_probs)
    if all_comp_probs
    else np.zeros((0, n_species), dtype=np.float32),
    species=np.array(competition_species),
)
import os

size_mb = os.path.getsize(OUT_NPZ) / 1e6
print(f"Saved {OUT_NPZ} ({size_mb:.1f} MB)")
print(f"  row_ids:   {len(all_row_ids)}")
if all_embeddings:
    emb_arr = np.stack(all_embeddings)
    print(f"  embeddings: {emb_arr.shape}")